In [ ]:
import csv
import os
import pickle
import sys
import time
from collections import defaultdict
from operator import itemgetter
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader, Dataset
from torch_geometric.nn.conv import MessagePassing
from tqdm import tqdm

from TKG import Quadruple, ReadableTKG, TemporalKnowledgeGraph
from logger import Logger
from profiling import EfficiencyProfiler
import utils

# Dataset

In [ ]:
class TKGDataset(Dataset):
    def __init__(self, dataset="YAGO", mode="train", history_len=5, dilate_len=1, add_inverse=True):
        super().__init__()
        assert mode in {"train", "valid", "test"}, f"Unsupported mode: {mode}"

        self.dataset_name = dataset
        self.mode = mode
        self.history_len = history_len
        self.dilate_len = dilate_len
        self.add_inverse = add_inverse

        self.data_path = os.path.join("./data", dataset)
        self.num_nodes, self.raw_num_rels, self.num_times = utils.get_total_number(self.data_path, "stat.txt")

        self.train_quads = self._load_split("train")
        self.valid_quads = self._load_split("valid")
        self.test_quads = self._load_split("test")

        self.train_graph = self._make_graph(self.train_quads, add_inverse=False)
        self.valid_graph = self._make_graph(self.valid_quads, add_inverse=False)
        self.test_graph = self._make_graph(self.test_quads, add_inverse=False)

        n_train = self.train_graph.num_snapshots
        n_valid = self.valid_graph.num_snapshots
        n_test = self.test_graph.num_snapshots

        if mode == "train":
            joint_quads = self.train_quads
            target_start = 0
            target_count = n_train
            warmup = history_len * dilate_len

        elif mode == "valid":
            joint_quads = self.train_quads + self.valid_quads
            target_start = n_train
            target_count = n_valid
            warmup = 0

        else:
            joint_quads = self.train_quads + self.valid_quads + self.test_quads
            target_start = n_train + n_valid
            target_count = n_test
            warmup = 0

        self.msg_graph = self._make_graph(joint_quads, add_inverse=add_inverse)
        self.raw_joint_graph = self._make_graph(joint_quads, add_inverse=False)

        self.joint_times = self._get_snapshot_times(joint_quads)
        self.graph = self.msg_graph
        self.num_rels = self.msg_graph.num_rels
        self.target_indices = list(range(target_start + warmup, target_start + target_count))
        self.times = self.target_indices

        if len(self.target_indices) == 0:
            raise ValueError(
                f"No usable samples for mode={mode}. "
                f"Check history_len={history_len}, dilate_len={dilate_len}, and split sizes."
            )

    def _load_split(self, split: str):
        data, _ = utils.load_quadruples(self.data_path, f"{split}.txt")
        return [Quadruple(int(r[0]), int(r[1]), int(r[2]), int(r[3])) for r in data]

    def _make_graph(self, quadruples, add_inverse: bool):
        return TemporalKnowledgeGraph(
            quadruples=quadruples,
            num_nodes=self.num_nodes,
            num_rels=self.raw_num_rels,
            add_inverse=add_inverse,
        )
    def _get_history_indices(self, target_idx):
        start = target_idx - self.history_len * self.dilate_len
        return list(range(start, target_idx, self.dilate_len))

    def _get_snapshot_times(self, quadruples):
        return sorted(set(int(q.tim) for q in quadruples))

    def __len__(self):
        return len(self.target_indices)

    def __getitem__(self, idx):
        target_idx = self.target_indices[idx]
    
        history = self.msg_graph.get_history(
            idx=target_idx,
            history_len=self.history_len,
            dilate_len=self.dilate_len,
        )

        target = self.msg_graph.get_snapshot(target_idx)
        target_quads = self.msg_graph.get_quadruples_at(target_idx)
    
        history_indices = self._get_history_indices(target_idx)
    
        history_times = [self.joint_times[i] for i in history_indices]
        target_time = self.joint_times[target_idx]
    
        return history, target, target_quads, history_times, target_time

    def collate_fn(self, batch):
        histories, targets, target_quads, history_times, target_times = zip(*batch)
        return (list(histories), list(targets), list(target_quads), list(history_times), list(target_times))

    def get_loader(self, batch_size=1, num_workers=0):
        return DataLoader(
            self,
            batch_size=batch_size,
            shuffle=(self.mode == "train"),
            num_workers=num_workers,
            collate_fn=self.collate_fn,
        )

    def stat(self):
        return self.graph.stat(f"{self.dataset_name}-{self.mode}")

    def __str__(self):
        return self.stat()

    def __repr__(self):
        return self.stat()


# Trainer

In [ ]:
def _as_quad_tuples(quadruples):
    if torch.is_tensor(quadruples):
        for row in quadruples.detach().cpu().tolist():
            yield tuple(map(int, row[:4]))
        return

    for quad in quadruples:
        if hasattr(quad, "sub") and hasattr(quad, "rel") and hasattr(quad, "obj") and hasattr(quad, "tim"):
            yield int(quad.sub), int(quad.rel), int(quad.obj), int(quad.tim)
        elif hasattr(quad, "src") and hasattr(quad, "rel") and hasattr(quad, "dst") and hasattr(quad, "tim"):
            yield int(quad.src), int(quad.rel), int(quad.dst), int(quad.tim)
        else:
            yield int(quad[0]), int(quad[1]), int(quad[2]), int(quad[3])


def _infer_raw_num_rels(dataset, add_inverse: bool) -> int:
    raw_num_rels = getattr(dataset, "raw_num_rels", None)
    if raw_num_rels is not None:
        return int(raw_num_rels)

    num_rels = int(getattr(dataset, "num_rels"))
    return num_rels // 2 if add_inverse else num_rels


def build_filter_answers_from_datasets(*datasets, add_inverse: bool = True):
    ent_answers = defaultdict(set)
    rel_answers = defaultdict(set)

    for dataset in datasets:
        raw_num_rels = _infer_raw_num_rels(dataset, add_inverse=add_inverse)

        quadruples = getattr(dataset, "graph", dataset).quadruples if hasattr(getattr(dataset, "graph", dataset), "quadruples") else dataset
        quad_list = list(_as_quad_tuples(quadruples))
        already_has_inverse = add_inverse and any(r >= raw_num_rels for _, r, _, _ in quad_list)

        for s, r, o, t in quad_list:
            # Add the query exactly as it appears in the source.
            ent_answers[(s, r, t)].add(o)
            rel_answers[(s, o, t)].add(r)

            # If the source is raw-only, add TiRGN-style inverse answers once.
            if add_inverse and (not already_has_inverse) and r < raw_num_rels:
                inv_r = r + raw_num_rels
                ent_answers[(o, inv_r, t)].add(s)
                rel_answers[(o, s, t)].add(inv_r)

    return ent_answers, rel_answers


def compute_raw_ranks(scores: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    _, indices = torch.sort(scores, dim=1, descending=True)
    matches = torch.nonzero(indices == targets.view(-1, 1), as_tuple=False)

    if matches.size(0) != targets.size(0):
        missing = targets.size(0) - matches.size(0)
        raise RuntimeError(f"Could not find {missing} target ids in sorted score indices. Check score shape and target ids.")

    ranks = torch.empty(targets.size(0), device=scores.device, dtype=torch.long)
    ranks[matches[:, 0]] = matches[:, 1] + 1
    return ranks.float()


def compute_filtered_ranks(scores: torch.Tensor, targets: torch.Tensor, filter_answers: list[set[int]] | None = None) -> torch.Tensor:
    if filter_answers is None:
        return compute_raw_ranks(scores, targets)

    filtered_scores = scores.clone()
    num_candidates = filtered_scores.size(1)

    for i, answers in enumerate(filter_answers):
        if not answers:
            continue

        target = int(targets[i].item())
        for ans in answers:
            ans = int(ans)
            if ans != target and 0 <= ans < num_candidates:
                filtered_scores[i, ans] = float("-inf")

    return compute_raw_ranks(filtered_scores, targets)


def compute_mrr(ranks: torch.Tensor) -> float:
    if ranks.numel() == 0:
        return 0.0
    return torch.mean(1.0 / ranks.float()).item()


def compute_hits_at_k(ranks: torch.Tensor, k: int) -> float:
    if ranks.numel() == 0:
        return 0.0
    return torch.mean((ranks <= k).float()).item()


def compute_ranking_metrics(ranks: torch.Tensor, ks=(1, 3, 10)) -> dict:
    metrics = {"mrr": compute_mrr(ranks)}
    for k in ks:
        metrics[f"hits@{k}"] = compute_hits_at_k(ranks, k)
    return metrics


In [ ]:
class Trainer:
    GRAD_CLIP_NORM = 3.0

    def __init__(
        self,
        model: torch.nn.Module,
        logger,
        model_config: dict,
        train_config: dict,
        dataset_config: dict,
        ent_weight: float = 0.7,
        rel_weight: float = 0.3,
        device: Optional[torch.device] = None,
    ):
        self.model = model
        self.logger = logger
        self.model_config = model_config
        self.train_config = train_config
        self.dataset_config = dataset_config
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.ent_weight = ent_weight
        self.rel_weight = rel_weight

        self.ent_loss_fn = torch.nn.CrossEntropyLoss()
        self.rel_loss_fn = torch.nn.CrossEntropyLoss()

        self.patience = train_config.get("patience", None)
        self.min_delta = train_config.get("min_delta", 0.0)
        self.num_epochs = train_config["max_epochs"]

        self.optimizer = self._build_optimizer()
        self.scheduler = self._build_scheduler()

        self.save_dir = logger.get_log_dir()
        self.chkpt_dir = os.path.join(self.save_dir, "chkpt")
        os.makedirs(self.chkpt_dir, exist_ok=True)

        self.epoch_losses: list[float] = []
        self.val_losses: list[float] = []

        self.model.to(self.device)
        self.efficiency_profiler = EfficiencyProfiler(
            model=self.model,
            device=self.device,
            forward_batch_fn=self._forward_batch,
            logger=self.logger,
        )

    def _build_optimizer(self):
        return torch.optim.Adam(
            self.model.parameters(),
            lr=self.train_config["lr"],
            weight_decay=self.train_config["weight_decay"],
        )

    def _build_scheduler(self):
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode="min",
            factor=self.train_config.get("lr_factor", 0.5),
            patience=self.train_config.get("lr_patience", 5),
            min_lr=self.train_config.get("min_lr", 1e-6),
        )

    def _prepare_batch(self, batch):
        history_graphs, targets, target_quads, history_times, target_times = batch
        history_graph = history_graphs[0]
        quads = target_quads[0].to(self.device).long()

        history_times = history_times[0] if history_times is not None else None
        target_times = target_times[0] if target_times is not None else None

        return history_graph, quads, history_times, target_times

    def _forward_batch(self, batch):
        history_graph, quads, history_times, target_times = self._prepare_batch(batch)
        ent_score, rel_score = self.model(history_graph, quads, history_times, target_times)
        return ent_score, rel_score, quads

    def _compute_loss(self, ent_score, rel_score, quads) -> torch.Tensor:
        ent_loss = self.ent_loss_fn(ent_score, quads[:, 2])
        rel_loss = self.rel_loss_fn(rel_score, quads[:, 1])
        return ent_loss + 0.1 * rel_loss

    def _step(self, batch, train: bool) -> float:
        if train:
            self.optimizer.zero_grad()

        ent_score, rel_score, quads = self._forward_batch(batch)
        loss = self._compute_loss(ent_score, rel_score, quads)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.GRAD_CLIP_NORM)
            self.optimizer.step()

        return loss.item()

    def _run_epoch(self, loader, desc: str, train: bool) -> float:
        self.model.train(train)

        total_loss = 0.0
        num_batches = 0

        iterator = tqdm(loader, desc=desc, leave=False, file=sys.stdout) if train else loader
        grad_context = torch.enable_grad() if train else torch.no_grad()

        with grad_context:
            for batch in iterator:
                total_loss += self._step(batch, train=train)
                num_batches += 1

        return total_loss / max(num_batches, 1)

    def _run_timed_epoch(self, loader, desc: str, train: bool):
        self._reset_peak_vram()

        start_time = time.time()
        loss = self._run_epoch(loader, desc=desc, train=train)
        elapsed_time = time.time() - start_time

        vram_msg = self._get_vram_msg("Train" if train else "Validation")

        return loss, elapsed_time, vram_msg

    def _reset_peak_vram(self) -> None:
        self.efficiency_profiler.vram.reset_peak()

    def _get_vram_msg(self, tag: str) -> Optional[str]:
        return self.efficiency_profiler.vram.get_message(tag)

    def _log_vram(self, tag: str) -> None:
        self.efficiency_profiler.vram.log(tag)

    def _log_vram_messages(self, *messages: Optional[str]) -> None:
        self.efficiency_profiler.vram.log_messages(*messages)

    def _profile_flops(self, loader) -> Optional[float]:
        return self.efficiency_profiler.flops.profile_single_forward(loader)

    def _measure_peak_vram(self, loader) -> Optional[float]:
        return self.efficiency_profiler.vram.measure_peak_single_batch(loader)

    def log_efficiency_stats(self, loader) -> None:
        self.efficiency_profiler.log_efficiency_stats(loader)

    def save_configs(self) -> None:
        config = {
            "model": self.model_config,
            "training": self.train_config,
            "dataset": self.dataset_config,
        }

        path = os.path.join(self.save_dir, "config.yaml")

        with open(path, "w") as f:
            yaml.dump(config, f, default_flow_style=False, sort_keys=False)

        self.logger.log_config(self.model_config, self.train_config, self.dataset_config)
        self.logger.log("SAVE", f"Config saved to {path}")

    def save_checkpoint(self, filename: str) -> None:
        path = os.path.join(self.chkpt_dir, filename)

        torch.save(
            {
                "model_state_dict": self.model.state_dict(),
                "optimizer_state_dict": self.optimizer.state_dict(),
                "epoch_losses": self.epoch_losses,
                "val_losses": self.val_losses,
            },
            path,
        )

        self.logger.log_checkpoint_saved(path)

    def load_checkpoint(self, path: str) -> None:
        checkpoint = torch.load(path, map_location=self.device)

        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        self.epoch_losses = checkpoint.get("epoch_losses", [])
        self.val_losses = checkpoint.get("val_losses", [])

        self.logger.log_checkpoint_loaded(path)

    def _step_scheduler(self, val_loss: float) -> None:
        old_lr = self.optimizer.param_groups[0]["lr"]

        self.scheduler.step(val_loss)

        new_lr = self.optimizer.param_groups[0]["lr"]

        if new_lr < old_lr:
            self.logger.log("INFO", f"Learning rate reduced: {old_lr} -> {new_lr}")

    def _update_best_checkpoint(self, val_loss: float, best_val_loss: float, bad_epochs: int):
        improved = val_loss < best_val_loss - self.min_delta

        if improved:
            self.save_checkpoint("best.pt")
            return val_loss, 0, True

        return best_val_loss, bad_epochs + 1, False

    def _should_stop_early(self, bad_epochs: int) -> bool:
        return self.patience is not None and bad_epochs >= self.patience

    def _log_training_summary(self, train_times: list[float], val_times: list[float]) -> None:
        avg_train_loss = sum(self.epoch_losses) / len(self.epoch_losses)
        avg_train_time = sum(train_times) / len(train_times)

        self.logger.log(
            "INFO",
            f"Avg Train Loss: {avg_train_loss:.6f} | Avg Train Time: {avg_train_time:.2f}s",
        )

        if self.val_losses:
            avg_val_loss = sum(self.val_losses) / len(self.val_losses)
            avg_val_time = sum(val_times) / len(val_times)
            self.logger.log(
                "INFO",
                f"Avg Val Loss: {avg_val_loss:.6f} | Avg Val Time: {avg_val_time:.2f}s",
            )

    def fit(self, train_loader, val_loader=None) -> dict[str, list[float]]:
        best_val_loss = float("inf")
        bad_epochs = 0

        train_times: list[float] = []
        val_times: list[float] = []

        fit_start_time = time.time()

        self.logger.log("INFO", f"Starting training for {self.num_epochs} epochs on device: {self.device}")

        for epoch in range(1, self.num_epochs + 1):
            train_loss, train_time, train_vram_msg = self._run_timed_epoch(
                train_loader,
                desc=f"[{epoch}/{self.num_epochs}] Train",
                train=True,
            )

            self.epoch_losses.append(train_loss)
            train_times.append(train_time)
            self.logger.log_train_epoch(epoch, train_loss, train_time)

            val_loss = None

            if val_loader is not None:
                val_loss, val_time, val_vram_msg = self._run_timed_epoch(
                    val_loader,
                    desc=f"[{epoch}/{self.num_epochs}] Val",
                    train=False,
                )

                self.val_losses.append(val_loss)
                val_times.append(val_time)
                self.logger.log_val_epoch(epoch, val_loss, val_time)

                self._log_vram_messages(train_vram_msg, val_vram_msg)
                self._step_scheduler(val_loss)

                best_val_loss, bad_epochs, _ = self._update_best_checkpoint(
                    val_loss,
                    best_val_loss,
                    bad_epochs,
                )

                if self._should_stop_early(bad_epochs):
                    elapsed_time = time.time() - fit_start_time
                    self.logger.log(
                        "INFO",
                        f"Stopped at epoch {epoch} | "
                        f"Best Val Loss: {best_val_loss:.6f} | "
                        f"No improvement for {bad_epochs} epochs | "
                        f"Elapsed Time: {elapsed_time:.2f}s",
                    )
                    break

            else:
                self._log_vram_messages(train_vram_msg)

            self.logger.log_metrics(epoch, train_loss, val_loss)

        self.save_checkpoint("final.pt")
        self.logger.log_training_completed()

        self._log_training_summary(train_times, val_times)

        return {
            "train": self.epoch_losses,
            "val": self.val_losses,
        }

    def _get_inverse_relation_offset(self) -> int:
        raw_num_rels = self.dataset_config.get("raw_num_rels")

        if raw_num_rels is not None:
            return int(raw_num_rels)

        if self.dataset_config.get("add_inverse", True):
            return int(self.model.num_rels // 2)

        return int(self.model.num_rels)

    def _add_inverse_quads_for_eval(self, quads: torch.Tensor) -> torch.Tensor:
        if not self.dataset_config.get("add_inverse", True):
            return quads

        if quads.numel() == 0:
            return quads

        rel_offset = self._get_inverse_relation_offset()
        max_input_rel_id = int(quads[:, 1].max().item())

        if max_input_rel_id >= self.model.num_rels:
            raise ValueError(
                f"Input relation id {max_input_rel_id} is outside model.num_rels={self.model.num_rels}. "
                "Check whether raw_num_rels/num_rels were configured correctly."
            )

        if max_input_rel_id >= rel_offset:
            return quads

        inverse_quads = quads[:, [2, 1, 0, 3]].clone()
        inverse_quads[:, 1] = inverse_quads[:, 1] + rel_offset

        all_quads = torch.cat((quads, inverse_quads), dim=0)

        max_rel_id = int(all_quads[:, 1].max().item())
        if max_rel_id >= self.model.num_rels:
            raise ValueError(
                f"Inverse relation id {max_rel_id} is outside model.num_rels={self.model.num_rels}. "
                "Make sure model.num_rels is the total relation count including inverse relations."
            )

        return all_quads

    def _get_filter_answers(self, quads, answers, key_fn):
        filter_answers = []

        for quad in quads:
            s = int(quad[0].item())
            r = int(quad[1].item())
            o = int(quad[2].item())
            t = int(quad[3].item())

            key = key_fn(s, r, o, t)
            filter_answers.append(answers.get(key, set()))

        return filter_answers

    def _get_entity_filter_answers(self, quads, ent_answers):
        return self._get_filter_answers(
            quads,
            ent_answers,
            key_fn=lambda s, r, o, t: (s, r, t),
        )

    def _get_relation_filter_answers(self, quads, rel_answers):
        return self._get_filter_answers(
            quads,
            rel_answers,
            key_fn=lambda s, r, o, t: (s, o, t),
        )

    def _compute_raw_and_filtered_ranks(self, scores, targets, filter_answers=None):
        raw_ranks = compute_raw_ranks(scores, targets)

        if filter_answers is None:
            filtered_ranks = raw_ranks
        else:
            filtered_ranks = compute_filtered_ranks(scores, targets, filter_answers)

        return raw_ranks, filtered_ranks

    @torch.no_grad()
    def _ranking_step(self, batch, ent_answers=None, rel_answers=None):
        history_graph, quads, history_times, target_times = self._prepare_batch(batch)
        eval_quads = self._add_inverse_quads_for_eval(quads)
        ent_score, rel_score = self.model(history_graph, eval_quads, history_times, target_times)

        ent_targets = eval_quads[:, 2]
        rel_targets = eval_quads[:, 1]

        ent_filter_answers = (
            self._get_entity_filter_answers(eval_quads, ent_answers)
            if ent_answers is not None
            else None
        )
        rel_filter_answers = (
            self._get_relation_filter_answers(eval_quads, rel_answers)
            if rel_answers is not None
            else None
        )

        ent_raw_ranks, ent_filter_ranks = self._compute_raw_and_filtered_ranks(
            ent_score,
            ent_targets,
            ent_filter_answers,
        )
        rel_raw_ranks, rel_filter_ranks = self._compute_raw_and_filtered_ranks(
            rel_score,
            rel_targets,
            rel_filter_answers,
        )

        return {
            "ent_raw": ent_raw_ranks.detach().cpu(),
            "ent_filter": ent_filter_ranks.detach().cpu(),
            "rel_raw": rel_raw_ranks.detach().cpu(),
            "rel_filter": rel_filter_ranks.detach().cpu(),
        }

    def _collect_ranks(self, test_loader, ent_answers=None, rel_answers=None):
        rank_buffers = {
            "ent_raw": [],
            "ent_filter": [],
            "rel_raw": [],
            "rel_filter": [],
        }

        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Test Ranking", leave=False, file=sys.stdout):
                ranks = self._ranking_step(
                    batch=batch,
                    ent_answers=ent_answers,
                    rel_answers=rel_answers,
                )

                for key in rank_buffers:
                    rank_buffers[key].append(ranks[key])

        return {key: torch.cat(values) for key, values in rank_buffers.items()}

    def _build_test_metrics(self, ranks, ks):
        return {
            "entity": {
                "raw": compute_ranking_metrics(ranks["ent_raw"], ks),
                "filter": compute_ranking_metrics(ranks["ent_filter"], ks),
            },
            "relation": {
                "raw": compute_ranking_metrics(ranks["rel_raw"], ks),
                "filter": compute_ranking_metrics(ranks["rel_filter"], ks),
            },
        }

    def _log_filter_metrics(self, name: str, metrics: dict) -> None:
        self.logger.log(
            "TEST",
            f"{name} Filter MRR: {metrics['mrr']:.6f} | "
            f"Hits@1: {metrics['hits@1']:.6f} | "
            f"Hits@3: {metrics['hits@3']:.6f} | "
            f"Hits@10: {metrics['hits@10']:.6f}",
        )

    def test(self, test_loader, ent_answers=None, rel_answers=None, ks=(1, 3, 10)) -> dict:
        self._reset_peak_vram()
        self.model.eval()

        start_time = time.time()
        ranks = self._collect_ranks(
            test_loader,
            ent_answers=ent_answers,
            rel_answers=rel_answers,
        )
        test_time = time.time() - start_time

        metrics = self._build_test_metrics(ranks, ks)
        metrics["test_time"] = test_time

        self.logger.log("TEST", f"Ranking test completed in {test_time:.2f}s")
        self._log_filter_metrics("Entity", metrics["entity"]["filter"])
        self._log_filter_metrics("Relation", metrics["relation"]["filter"])
        self._log_vram("Test")

        return metrics

# Model

In [ ]:
class RGCNConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        num_relations,
        num_bases=None,
        relation_dim=32,
        dynamic_num_bases=4,
        dropout=0.0,
        use_relation_convolution=True
    ):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_relations = num_relations
        self.num_bases = num_bases
        self.relation_dim = relation_dim
        self.dynamic_num_bases = dynamic_num_bases
        self.use_relation_convolution = use_relation_convolution

        self.dropout = nn.Dropout(dropout)
        self.use_basis = num_bases is not None and 0 < num_bases < num_relations

        if self.use_basis:
            self.basis = nn.Parameter(torch.empty(num_bases, in_channels, out_channels))
            self.att = nn.Parameter(torch.empty(num_relations, num_bases))
        else:
            self.weight = nn.Parameter(torch.empty(num_relations, in_channels, out_channels))

        self.root = nn.Parameter(torch.empty(in_channels, out_channels))
        self.bias = nn.Parameter(torch.empty(out_channels))

        if self.use_relation_convolution:
            self.rel_norm = nn.LayerNorm(relation_dim)

            self.dynamic_basis = nn.Parameter(torch.empty(dynamic_num_bases, in_channels, out_channels))
            self.rel_to_coeff = nn.Sequential(
                nn.LayerNorm(relation_dim), 
                nn.Linear(relation_dim, 64), 
                nn.GELU(), 
                nn.Dropout(dropout), 
                nn.Linear(64, dynamic_num_bases)
            )
            self.rel_gate = nn.Sequential(nn.LayerNorm(relation_dim), nn.Linear(relation_dim, 1))
            self.dynamic_scale = nn.Parameter(torch.tensor(0.1))

            self.rel_film = nn.Sequential(nn.LayerNorm(relation_dim), nn.Linear(relation_dim, 2 * out_channels))
            self.rel_to_node = nn.Linear(relation_dim, in_channels)

        self.reset_parameters()

    def reset_parameters(self):
        if self.use_basis:
            nn.init.xavier_uniform_(self.basis)
            nn.init.xavier_uniform_(self.att)
        else:
            nn.init.xavier_uniform_(self.weight)

        nn.init.xavier_uniform_(self.root)
        nn.init.zeros_(self.bias)

        if self.use_relation_convolution:
            # nn.init.xavier_uniform_(self.rel_emb.weight)
            nn.init.xavier_uniform_(self.dynamic_basis)

            # for module in self.rel_to_coeff:
            #     if isinstance(module, nn.Linear):
            #         nn.init.xavier_uniform_(module.weight)
                    # nn.init.zeros_(module.bias)

            nn.init.zeros_(self.rel_to_coeff[-1].weight)
            nn.init.zeros_(self.rel_to_coeff[-1].bias)

            # nn.init.xavier_uniform_(self.rel_gate[1].weight)
            nn.init.zeros_(self.rel_gate[1].bias)

            nn.init.zeros_(self.rel_film[1].weight)
            nn.init.zeros_(self.rel_film[1].bias)

            # nn.init.xavier_uniform_(self.rel_to_node.weight)
            nn.init.zeros_(self.rel_to_node.bias)

    def get_relation_state(self, relation_state=None):
        if not self.use_relation_convolution:
            return None

        return self.rel_norm(relation_state)

    def relation_weight(self, rel_h=None):
        if self.use_basis:
            W = torch.einsum("rb,bio->rio", self.att, self.basis)
        else:
            W = self.weight

        if self.use_relation_convolution:
            coeff = torch.tanh(self.rel_to_coeff(rel_h))
            delta_W = torch.einsum("rb,bio->rio", coeff, self.dynamic_basis)
            gate = torch.sigmoid(self.rel_gate(rel_h)).view(self.num_relations, 1, 1)
            W = W + gate * delta_W

        return W

    def compute_edge_norm(self, edge_index, edge_type, num_nodes):
        dst = edge_index[1]
        key = edge_type * num_nodes + dst
        deg = torch.zeros(self.num_relations * num_nodes, device=edge_index.device, dtype=torch.float32)
        deg.index_add_(0, key, torch.ones_like(key, dtype=torch.float32))
        edge_norm = 1.0 / deg[key].clamp(min=1.0)
        return edge_norm

    def forward(self, x, edge_index, edge_type, relation_state=None, edge_norm=None):
        edge_index = edge_index.long()
        edge_type = edge_type.long().view(-1)

        src = edge_index[0]
        dst = edge_index[1]
        num_nodes = x.size(0)

        rel_h = self.get_relation_state(relation_state)
        W = self.relation_weight(rel_h)

        x_src = x[src]

        if self.use_relation_convolution:
            rel_node = torch.tanh(self.rel_to_node(rel_h))
            x_src = x_src * rel_node[edge_type]

        msg = torch.einsum("ei,eio->eo", x_src, W[edge_type])

        if self.use_relation_convolution:
            gamma, beta = self.rel_film(rel_h).chunk(2, dim=-1)
            gamma = torch.tanh(gamma)
            msg = msg * (1.0 + gamma[edge_type]) + beta[edge_type]

        if edge_norm is None:
            edge_norm = self.compute_edge_norm(edge_index, edge_type, num_nodes)

        edge_norm = edge_norm.to(device=msg.device, dtype=msg.dtype)
        msg = msg * edge_norm.view(-1, 1)
        msg = self.dropout(msg)

        out = x.new_zeros(num_nodes, self.out_channels)
        out.index_add_(0, dst, msg)
        out = out + x @ self.root
        out = out + self.bias

        return out

In [ ]:
class ConvTransBase(torch.nn.Module):
    def __init__(self, num_b, embedding_dim, input_dropout=0, hidden_dropout=0, feature_map_dropout=0, channels=50, kernel_size=3):
        super().__init__()
        self.inp_drop = torch.nn.Dropout(input_dropout)
        self.hidden_drop = torch.nn.Dropout(hidden_dropout)
        self.feature_map_drop = torch.nn.Dropout(feature_map_dropout)
        self.conv1 = torch.nn.Conv1d(2, channels, kernel_size, stride=1, padding=kernel_size // 2)
        self.bn0 = torch.nn.BatchNorm1d(2)
        self.bn1 = torch.nn.BatchNorm1d(channels)
        self.bn2 = torch.nn.BatchNorm1d(embedding_dim)
        self.register_parameter('b', nn.Parameter(torch.zeros(num_b)))
        self.fc = torch.nn.Linear(embedding_dim * channels, embedding_dim)

    def conv_pipeline(self, x, batch_size):
        x = self.inp_drop(self.bn0(x))
        x = self.feature_map_drop(F.relu(self.bn1(self.conv1(x))))
        x = self.hidden_drop(self.fc(x.view(batch_size, -1)))
        x = F.relu(self.bn2(x) if batch_size > 1 else x)
        return x


class ConvTransR(ConvTransBase):
    def __init__(self, num_relations, embedding_dim, input_dropout=0, hidden_dropout=0, feature_map_dropout=0, channels=50, kernel_size=3):
        super().__init__(num_relations * 2, embedding_dim, input_dropout, hidden_dropout, feature_map_dropout, channels, kernel_size)

    def forward(self, embedding, emb_rel, triplets):
        e1_embedded_all = F.tanh(embedding)
        e1 = e1_embedded_all[triplets[:, 0]].unsqueeze(1)
        e2 = e1_embedded_all[triplets[:, 2]].unsqueeze(1)
        x = self.conv_pipeline(torch.cat([e1, e2], 1), len(triplets))
        return torch.mm(x, emb_rel.transpose(1, 0))


class ConvTransE(ConvTransBase):
    def __init__(self, num_entities, embedding_dim, input_dropout=0, hidden_dropout=0, feature_map_dropout=0, channels=50, kernel_size=3):
        super().__init__(num_entities, embedding_dim, input_dropout, hidden_dropout, feature_map_dropout, channels, kernel_size)

    def forward(self, embedding, emb_rel, triplets, partial_embeding=None):
        e1_embedded_all = F.tanh(embedding)
        e1 = e1_embedded_all[triplets[:, 0]].unsqueeze(1)
        rel = emb_rel[triplets[:, 1]].unsqueeze(1)
        x = self.conv_pipeline(torch.cat([e1, rel], 1), len(triplets))
        target = e1_embedded_all if partial_embeding is None else partial_embeding
        return torch.mm(x, target.transpose(1, 0))

    def forward_slow(self, embedding, emb_rel, triplets):
        e1_embedded_all = F.tanh(embedding)
        e1 = e1_embedded_all[triplets[:, 0]].unsqueeze(1)
        rel = emb_rel[triplets[:, 1]].unsqueeze(1)
        x = self.conv_pipeline(torch.cat([e1, rel], 1), len(triplets))
        return torch.sum(torch.mul(x, e1_embedded_all[triplets[:, 2]]), dim=1)

In [ ]:
class GatedResidualCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.hidden_proj = nn.Linear(hidden_dim, hidden_dim)
        self.gate = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, h):
        candidate = F.tanh(self.input_proj(x) + self.hidden_proj(h))
        z = F.sigmoid(self.gate(torch.cat([x, h], dim=-1)))

        out = z * h + (1.0 - z) * candidate
        out = self.dropout(out)
        return self.norm(out)

In [ ]:
class TimeGapTransitionGate(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.0, use_l2_norm: bool = True):
        super().__init__()
        self.use_l2_norm = use_l2_norm

        self.transition = nn.Parameter(torch.zeros(1, dim))
        self.gate = nn.Sequential(
            nn.LayerNorm(dim * 3),
            nn.Linear(dim * 3, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )

    def _normalize(self, x):
        if self.use_l2_norm:
            return F.normalize(x, p=2, dim=-1)
        return x

    def forward(self, prev_state, new_state):
        base_transition = self.transition.expand(new_state.size(0), -1)
        gate_input = torch.cat([prev_state, new_state, base_transition], dim=-1)
        z = self.gate(gate_input)
        out = z * prev_state + (1.0 - z) * new_state
        return self._normalize(out)

In [ ]:
class SnapshotEvolutionBlock(nn.Module):
    def __init__(self, num_rels: int, dim: int, num_rgcn_layers: int = 1, dropout: float = 0.0, use_l2_norm: bool = True, num_bases: Optional[int] = None):
        super().__init__()
        self.num_rels = num_rels
        self.dim = dim
        self.use_l2_norm = use_l2_norm

        self.rgcn_layers = nn.ModuleList([
            RGCNConv(dim, dim, num_rels, num_bases=num_bases, dropout=dropout)
            for _ in range(num_rgcn_layers)
        ])

        self.relation_cell = GatedResidualCell(dim * 2, dim, dropout=dropout)
        self.entity_cell = GatedResidualCell(dim, dim, dropout=dropout)

    def _normalize(self, x):
        if self.use_l2_norm:
            return F.normalize(x, p=2, dim=1)
        return x

    def _relation_context(self, entity_state, edge_index, edge_type):
        rel_context = entity_state.new_zeros(self.num_rels, self.dim)
        rel_count = entity_state.new_zeros(self.num_rels, 1)

        if edge_type.numel() == 0:
            return rel_context

        src, dst = edge_index
        rel_ids = torch.cat([edge_type, edge_type], dim=0)
        node_ids = torch.cat([src, dst], dim=0)

        rel_context.index_add_(0, rel_ids, entity_state[node_ids])

        ones = torch.ones(rel_ids.size(0), 1, device=entity_state.device, dtype=entity_state.dtype)
        rel_count.index_add_(0, rel_ids, ones)

        return rel_context / rel_count.clamp(min=1.0)

    def _evolve_relation(self, init_rel_emb, relation_state, entity_state, edge_index, edge_type):
        rel_context = self._relation_context(entity_state, edge_index, edge_type)
        relation_input = torch.cat([init_rel_emb, rel_context], dim=1)
        relation_state = self.relation_cell(relation_input, relation_state)
        return self._normalize(relation_state)

    def _evolve_entity(self, entity_state, relation_state, edge_index, edge_type):
        h = entity_state

        for rgcn_layer in self.rgcn_layers:
            h = rgcn_layer(h, edge_index, edge_type, relation_state)
            h = F.gelu(h)

        entity_state = self.entity_cell(h, entity_state)
        return self._normalize(entity_state)

    def forward(self, init_rel_emb, entity_state, relation_state, edge_index, edge_type):
        relation_state = self._evolve_relation(init_rel_emb, relation_state, entity_state, edge_index, edge_type)
        entity_state = self._evolve_entity(entity_state, relation_state, edge_index, edge_type)

        return entity_state, relation_state

In [ ]:
class EvolutionModule(nn.Module):
    def __init__(self, num_nodes: int, num_rels: int, dim: int, num_times: int, num_rgcn_layers: int = 1, dropout: float = 0.0, use_l2_norm: bool = True, num_bases: Optional[int] = None):
        super().__init__()
        self.num_nodes = num_nodes
        self.num_rels = num_rels
        self.dim = dim
        self.num_times = num_times
        self.use_l2_norm = use_l2_norm

        self.evolution_blocks = nn.ModuleList([
            SnapshotEvolutionBlock(num_rels, dim, num_rgcn_layers, dropout, use_l2_norm, num_bases)
            for _ in range(num_times)
        ])

        self.entity_time_gate = TimeGapTransitionGate(dim, dropout=dropout, use_l2_norm=use_l2_norm)
        self.relation_time_gate = TimeGapTransitionGate(dim, dropout=dropout, use_l2_norm=use_l2_norm)

    def _snapshot_edges(self, snapshot, device):
        edge_index = snapshot.edge_index.to(device).long()
        edge_type = snapshot.edge_attr.to(device).long().view(-1)
        return edge_index, edge_type

    def forward(self, history_graphs, init_ent_emb, init_rel_emb, history_times, return_all: bool = True):
        device = init_ent_emb.device

        entity_state = init_ent_emb
        relation_state = init_rel_emb

        history_entity_states = []
        history_relation_states = []

        for i, snapshot in enumerate(history_graphs):
            edge_index, edge_type = self._snapshot_edges(snapshot, device)
            prev_entity_state = entity_state
            prev_relation_state = relation_state
            entity_state, relation_state = self.evolution_blocks[i](init_rel_emb, entity_state, relation_state, edge_index, edge_type)

            if history_times is not None and i > 0:
                relation_state = self.relation_time_gate(prev_relation_state, relation_state)
                entity_state = self.entity_time_gate(prev_entity_state, entity_state)

            if return_all:
                history_entity_states.append(entity_state)
                history_relation_states.append(relation_state)

        return entity_state, relation_state, history_entity_states, history_relation_states

In [ ]:
class Model(nn.Module):
    def __init__(
        self,
        num_nodes: int,
        num_rels: int,
        embedding_dim: int,
        L: int,
        num_times: int,
        device: str = "cuda",
        dropout: float = 0.2,
        encoder_layers: int = 1,
        use_l2_norm: bool = True,
        input_dropout: Optional[float] = None,
        hidden_dropout: Optional[float] = None,
        feature_map_dropout: Optional[float] = None,
        channels: int = 50,
        kernel_size: int = 3,
        num_bases: Optional[int] = None,
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.num_rels = num_rels
        self.embedding_dim = embedding_dim
        self.L = L
        self.device = device

        input_dropout = dropout if input_dropout is None else input_dropout
        hidden_dropout = dropout if hidden_dropout is None else hidden_dropout
        feature_map_dropout = dropout if feature_map_dropout is None else feature_map_dropout

        self.ent_embedding = nn.Parameter(torch.empty(num_nodes, embedding_dim))
        self.rel_embedding = nn.Parameter(torch.empty(num_rels, embedding_dim))

        self.evolution = EvolutionModule(
            num_nodes=num_nodes,
            num_rels=num_rels,
            dim=embedding_dim,
            num_rgcn_layers=encoder_layers,
            dropout=dropout,
            use_l2_norm=use_l2_norm,
            num_bases=num_bases,
            num_times=L,
        )

        self.ent_decoder = ConvTransE(
            num_nodes,
            embedding_dim,
            input_dropout=input_dropout,
            hidden_dropout=hidden_dropout,
            feature_map_dropout=feature_map_dropout,
            channels=channels,
            kernel_size=kernel_size,
        )

        self.rel_decoder = ConvTransR(
            num_rels,
            embedding_dim,
            input_dropout=input_dropout,
            hidden_dropout=hidden_dropout,
            feature_map_dropout=feature_map_dropout,
            channels=channels,
            kernel_size=kernel_size,
        )

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.ent_embedding)
        torch.nn.init.xavier_uniform_(self.rel_embedding)

    def get_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, history_graphs, triples, history_times, target_times):
        device = self.ent_embedding.device
        triples = triples.to(device).long()
    
        ent_state, rel_state, history_ent_states, history_rel_states = self.evolution(
            history_graphs,
            self.ent_embedding,
            self.rel_embedding,
            history_times,
            return_all=True,
        )

        ent_score = self.ent_decoder(ent_state, rel_state, triples)
        rel_score = self.rel_decoder(ent_state, rel_state, triples)
    
        return ent_score, rel_score

In [ ]:
model_config = {
    'embedding_dim': 32,
    'dropout': 0.5,
}

train_config = {
    # 'dataset_name': 'ICEWS18',  # Change to: 'GDELT', 'ICEWS14', 'ICEWS18', 'WIKI'
    'batch_size': 1,
    'max_epochs': 100,
    'lr': 0.001,
    'weight_decay': 1e-5,
    'save_dir': 'SAVE',
    "patience": 10,
    "min_delta": 0.0001,
    "lr_patience": 5,
    "lr_factor": 0.1,
    "min_lr": 0.000001,
}

dataset_config = {
    "dataset": "YAGO",
    "history_len": 2,
    "dilate_len": 1,
    "add_inverse": True
}

In [ ]:
train_dataset = TKGDataset(**dataset_config, mode="train")
valid_dataset = TKGDataset(**dataset_config, mode="valid")
test_dataset = TKGDataset(**dataset_config, mode="test")

dataset_config["raw_num_rels"] = train_dataset.raw_num_rels
dataset_config["num_times"] = train_dataset.num_times

In [ ]:
train_loader = train_dataset.get_loader(batch_size=train_config["batch_size"])
val_loader = valid_dataset.get_loader(batch_size=train_config["batch_size"])
test_loader = test_dataset.get_loader(batch_size=train_config["batch_size"])

In [ ]:
len(valid_dataset), len(test_dataset)

In [ ]:
device = f'cuda' if torch.cuda.is_available() else 'cpu'
model = Model(
    train_dataset.num_nodes,
    train_dataset.num_rels,
    embedding_dim=model_config['embedding_dim'],
    L=dataset_config['history_len'],
    num_times=dataset_config['num_times']
)

model = model.to(device)

num_trainable_params = model.get_params()
print(f"Number of trainable parameters: {num_trainable_params:,}")

In [ ]:
logger = Logger(save_dir=train_config['save_dir'])

train_loader = train_dataset.get_loader(batch_size=train_config['batch_size'])
val_loader = valid_dataset.get_loader(batch_size=train_config['batch_size'])
test_loader = test_dataset.get_loader(batch_size=train_config['batch_size'])

trainer = Trainer(model, logger, model_config, train_config, dataset_config)
trainer.save_configs()
# logger.log("INFO", f"Number of trainable parameters: {model.get_params():,}")
trainer.log_efficiency_stats(train_loader)
history = trainer.fit(train_loader, val_loader)

# Load best validation checkpoint before testing
best_ckpt_path = os.path.join(trainer.chkpt_dir, "best.pt")
if os.path.exists(best_ckpt_path):
    trainer.load_checkpoint(best_ckpt_path)
    logger.log("INFO", f"Loaded best checkpoint for test: {best_ckpt_path}")
else:
    logger.log("WARN", f"Best checkpoint not found; testing current model instead: {best_ckpt_path}")

ent_answers, rel_answers = build_filter_answers_from_datasets(test_dataset, add_inverse=True)
test_metrics = trainer.test(test_loader, ent_answers=ent_answers, rel_answers=rel_answers)

logger.close()

In [ ]:
plt.plot(history['train'], label='Train Loss')
plt.plot(history['val'], label='Train Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss over Iterations')
plt.grid()
plt.show()